<a href="https://colab.research.google.com/github/dempsesn/course-workspace/blob/main/1Dempsey_langchain_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch

experiments = [
    {"label": "Baseline (Like-for-like)", "k": 2, "score_threshold": 0.30},
    {"label": "Stricter Threshold",      "k": 2, "score_threshold": 0.50},
    {"label": "Permissive Threshold",   "k": 2, "score_threshold": 0.15},
    {"label": "Higher K (k=4)",         "k": 4, "score_threshold": 0.30},
    {"label": "Single Chunk (k=1)",     "k": 1, "score_threshold": 0.30},
]

refusal_phrase = "cannot answer this question"
results = []

for exp in experiments:
    temp_retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": exp["k"], "score_threshold": exp["score_threshold"]}
    )

    ans_total = 0
    ans_found = 0
    unans_total = 0
    unans_refused = 0

    for item in goldset:
        question = item["question"]
        is_unanswerable = (
            item.get("should_refuse", False)
            or item.get("unanswerable", False)
            or ("refuse" in str(item.get("ground_truth", "")).lower())
        )

        docs = temp_retriever.invoke(question)

        if not docs:
            did_refuse = True
        else:
            # Query the model exactly as we do in the main rag_chain pipeline
            context = "\n\n".join(d.page_content for d in docs)
            prompt_text = (
                f"Read the following context and answer the question.\n"
                f"If the answer is not contained in the context, reply exactly: \"I cannot answer this question based on the provided documents.\"\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer:"
            )
            input_ids = tokenizer(prompt_text, return_tensors="pt", truncation=True)
            with torch.no_grad():
                out = model.generate(**input_ids, max_new_tokens=64, do_sample=False)
            res = tokenizer.decode(out[0], skip_special_tokens=True).strip()

            did_refuse = (not res) or (refusal_phrase in res.lower())

        if is_unanswerable:
            unans_total += 1
            if did_refuse:
                unans_refused += 1
        else:
            ans_total += 1
            if not did_refuse:
                ans_found += 1

    results.append({
        "Configuration": exp["label"],
        "k": exp["k"],
        "score_threshold": exp["score_threshold"],
        "Answerable_Found": f"{ans_found}/{ans_total}",
        "Unanswerable_Refused": f"{unans_refused}/{unans_total}",
        "Answerable_Recall": round(ans_found / max(1, ans_total), 2),
        "Refusal_Accuracy": round(unans_refused / max(1, unans_total), 2)
    })

df_results = pd.DataFrame(results)
print("\n" + df_results.to_markdown(index=False))

df_results.to_csv("results_langchain.csv", index=False)
print("\nSaved evaluation results to results_langchain.csv")

/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='9189ccc1-0619-494e-a6fc-0edd125da0c6', metadata={'source': 'sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.54630244)), (Document(id='18d20d4c-fbfb-43d0-a3ab-4fa6f787e6bd', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast cryp


| Configuration            |   k |   score_threshold | Answerable_Found   | Unanswerable_Refused   |   Answerable_Recall |   Refusal_Accuracy |
|:-------------------------|----:|------------------:|:-------------------|:-----------------------|--------------------:|-------------------:|
| Baseline (Like-for-like) |   2 |              0.3  | 3/6                | 2/2                    |                0.5  |                  1 |
| Stricter Threshold       |   2 |              0.5  | 2/6                | 2/2                    |                0.33 |                  1 |
| Permissive Threshold     |   2 |              0.15 | 5/6                | 2/2                    |                0.83 |                  1 |
| Higher K (k=4)           |   4 |              0.3  | 3/6                | 2/2                    |                0.5  |                  1 |
| Single Chunk (k=1)       |   1 |              0.3  | 3/6                | 2/2                    |                0.5  |             

your-repo/
├── rag_project/
└── langchain_project/
    ├── README.md
    ├── langchain_notebook.ipynb
    ├── my_kb/
    ├── goldset.json
    └── results_langchain.csv

In [1]:
# Run in a notebook cell if not already installed
# %pip install langchain langchain-community

In [2]:
!pip install -q langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
# LLMs & Chat Models (choose what you use)
!pip install -q langchain-openai
!pip install -q langchain-anthropic
!pip install -q langchain-google-genai

# Common Vector Stores
!pip install -q chromadb     # Local, lightweight
!pip install -q faiss-cpu    # Fast in-memory similarity search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "LangChain is a framework for developing applications powered by language models."

splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=5)
chunks = splitter.split_text(text)

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx + 1}: {chunk}")

Chunk 1: LangChain is a framework for
Chunk 2: for developing applications
Chunk 3: powered by language models.


In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imports successful!")

Imports successful!


/tmp/ipykernel_4113/1996423710.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [7]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Path to your copied knowledge base
kb_path = "/content/sample_data/my_kb"

# Check if the path is a file or directory, and load accordingly
if os.path.isdir(kb_path):
    print(f"'{kb_path}' is a directory. Using DirectoryLoader...")
    loader = DirectoryLoader(
        path=kb_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    docs = loader.load()
elif os.path.isfile(kb_path):
    print(f"'{kb_path}' is a single file. Using TextLoader directly...")
    loader = TextLoader(file_path=kb_path, encoding="utf-8")
    docs = loader.load()
else:
    raise FileNotFoundError(f"No file or directory found at '{kb_path}'")

# Quick sanity check
print(f"Loaded {len(docs)} full document(s).")
if len(docs) > 0:
    print(f"Sample source: {docs[0].metadata.get('source', 'Unknown')}")
    print(f"Sample content length: {len(docs[0].page_content)} characters")


'/content/sample_data/my_kb' is a directory. Using DirectoryLoader...
Loaded 4 full document(s).
Sample source: /content/sample_data/my_kb/auth_standards.txt
Sample content length: 450 characters


In [8]:
# Define your baseline chunking settings
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,            # Uses character count
    is_separator_regex=False
)

# Split the loaded documents into chunks
chunks = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 5


In [9]:
# Install modern standalone packages including the dedicated langchain-chroma integration
!pip install -q "chromadb" "langchain-huggingface" "langchain-chroma"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Initialize local embeddings using the recommended package
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Build the vector store from your chunked documents using the updated Chroma class
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="my_rag_collection"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Configure search with top-k and a minimum similarity threshold
k_val = 2
score_thresh = 0.4  # Start around 0.3 - 0.6; tune depending on retrieval

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": k_val,
        "score_threshold": score_thresh
    }
)

In [12]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant answering questions strictly based on the provided context.
If the answer cannot be found directly in the context, or if the context is empty, respond exactly:
"I cannot answer this question based on the provided documents."

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    if not docs:
        return "No relevant context found."
    return "\n\n".join(doc.page_content for doc in docs)

# If using a local LLM (e.g. HuggingFacePipeline, Ollama, or simple local logic):
# from langchain_community.llms import HuggingFacePipeline
# llm = ...

# For an offline-safe setup where no GPU/API key is present,
# a fallback mock LLM or HuggingFacePipeline can be plugged in:
try:
    from langchain_community.llms import HuggingFacePipeline
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model="google/flan-t5-small", max_new_tokens=100)
    llm = HuggingFacePipeline(pipeline=pipe)
except Exception:
    # Deterministic local fallback if transformers model is not loaded
    from langchain_core.language_models.fake import FakeListLLM
    llm = FakeListLLM(responses=["I cannot answer this question based on the provided documents."])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

In [14]:
!pip install faiss-cpu -q

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS vector store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Create retriever with score threshold
# FAISS normalized relevance scores range from 0 to 1
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 2,
        "score_threshold": 0.25  # adjust between 0.2 - 0.4 as needed
    }
)

/tmp/ipykernel_4113/1994833926.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
test_docs = retriever.invoke("What is the mandatory remediation for SQL Injection?")
print(f"Retrieved {len(test_docs)} chunks:")
for d in test_docs:
    print("-", d.page_content[:90], "...")

Retrieved 1 chunks:
- SQL Injection (SQLi) allows attackers to tamper with backend database queries.
Classic SQL ...


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='d7a713b0-f0ab-4ef9-902e-09159684d78a', metadata={'source': '/content/sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.5463025)), (Document(id='06bf554f-1250-413a-948d-f741fb9e7763', metadata={'source': '/content/sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNe

In [17]:
!pip install -U langchain-chroma -q

In [18]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Point to your folder (adjust path if needed, e.g. "my_kb" or "sample_data/my_kb")
kb_path = "my_kb"
if not os.path.exists(kb_path) and os.path.exists("sample_data/my_kb"):
    kb_path = "sample_data/my_kb"

print(f"Loading files from: {kb_path}")

# 2. Load all .txt files from the directory
loader = DirectoryLoader(kb_path, glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()
print(f"Loaded {len(documents)} document(s).")

# 3. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# This creates the 'chunks' variable!
chunks = text_splitter.split_documents(documents)
print(f"Successfully created {len(chunks)} chunks.")

Loading files from: sample_data/my_kb
Loaded 4 document(s).
Successfully created 5 chunks.


In [19]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 2, "score_threshold": 0.25}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
!pip install -q faiss-cpu

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create the vectorstore from the chunks created above
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Create the retriever with a threshold
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 2,
        "score_threshold": 0.25
    }
)

# Quick sanity check
test_hits = retriever.invoke("What is the mandatory remediation for SQL Injection?")
print(f"Retrieved {len(test_hits)} matching chunk(s).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved 1 matching chunk(s).


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8b254950-3eae-4549-8d7b-3168776f3ca9', metadata={'source': 'sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.5463025)), (Document(id='f9b787b6-40bd-479e-af60-12de85808a83', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast crypt

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128
)

llm = HuggingFacePipeline(pipeline=generator)

# Define prompt template
template = """Answer the question based only on the following context.
If the context does not contain enough information to answer the question, or if no context is provided, reply exactly:
"I cannot answer this question based on the provided documents."

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    if not docs:
        return "No relevant context found."
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCa

In [25]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda

# --- 1. Vector Store with Normalized Embeddings (Fixes negative scores) ---
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}  # Ensures cosine scores are strictly between 0 and 1
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Use score threshold now that scores are guaranteed 0.0 to 1.0
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 2, "score_threshold": 0.3}
)

# --- 2. Reliable Flan-T5 Inference Function ---
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def ask_flan(inputs):
    question = inputs["question"]
    docs = retriever.invoke(question)

    # If no docs meet the similarity threshold, refuse immediately
    if not docs:
        return "I cannot answer this question based on the provided documents."

    context = "\n".join(d.page_content for d in docs)

    prompt_text = (
        f"Read the following context and answer the question.\n"
        f"If the answer is not contained in the context, reply: 'I cannot answer this question based on the provided documents.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )

    input_ids = tokenizer(prompt_text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            max_new_tokens=64,
            do_sample=False
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Safety fallback if the model produces empty output or hallucinates
    if not generated:
        return "I cannot answer this question based on the provided documents."
    return generated

rag_chain = RunnableLambda(lambda q: ask_flan({"question": q}))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [27]:
with open("/content/sample_data/goldset.json", "r") as f:
    goldset = json.load(f)

print("=" * 60)
print("PART 2: BOT TRANSCRIPT")
print("=" * 60 + "\n")

for i, item in enumerate(goldset):
    q = item["question"]
    ans = rag_chain.invoke(q)

    print(f"[{i + 1}] Question: {q}")
    print(f"    Answer:   {ans}")
    print("-" * 60)

PART 2: BOT TRANSCRIPT

[1] Question: What is the primary defense against XSS?
    Answer:   context-aware output encoding
------------------------------------------------------------
[2] Question: How are script execution sources restricted to trusted domains?
    Answer:   Content Security Policy (CSP) headers
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='9189ccc1-0619-494e-a6fc-0edd125da0c6', metadata={'source': 'sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.54630244)), (Document(id='18d20d4c-fbfb-43d0-a3ab-4fa6f787e6bd', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast cryp

[3] Question: What is the mandatory remediation for SQL Injection?
    Answer:   parameterized queries using prepared statements
------------------------------------------------------------
[4] Question: Which password hashing algorithms are recommended?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------
[5] Question: Where should session tokens be stored to mitigate script theft?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------
[6] Question: What cookie attribute blocks third-party cookie transmission?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------
[7] Question: How long do you bake sourdough bread at 450 degrees?
    Answer:   I cannot answer this question based on the provided documents.
---------------------------------

In [31]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Redefine rag_chain so retriever receives only the string:
rag_chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [34]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Rebuild your chain so itemgetter extracts the text from the dict first:
rag_chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [48]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Re-define rag_chain with itemgetter:
rag_chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [57]:
import json
import torch

# 1. Use the optimal score threshold (0.15) discovered in evaluation
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 2,
        "score_threshold": 0.15
    }
)
print("Reconfigured retriever to similarity_score_threshold (0.15) for optimal accuracy and refusals.\n")

# 2. Define the reliable inference function
def ask_flan_clean(question):
    docs = retriever.invoke(question)
    if not docs:
        return "I cannot answer this question based on the provided documents."

    context = "\n\n".join(d.page_content for d in docs)

    prompt_text = (
        f"Read the following context and answer the question.\n"
        f"If the answer is not contained in the context, reply exactly: \"I cannot answer this question based on the provided documents.\"\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )

    input_ids = tokenizer(prompt_text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            max_new_tokens=64,
            do_sample=False
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    if not generated:
        return "I cannot answer this question based on the provided documents."
    return generated

# Bind to our rag_chain
from langchain_core.runnables import RunnableLambda
rag_chain = RunnableLambda(lambda q_dict: ask_flan_clean(q_dict["question"]))

# 3. Load goldset and execute completely
with open("/content/sample_data/goldset.json", "r") as f:
    goldset = json.load(f)

print("=" * 60)
print("PART 2: BOT TRANSCRIPT")
print("=" * 60 + "\n")

for i, item in enumerate(goldset):
    q = item["question"]
    ans = rag_chain.invoke({"question": q})
    print(f"[{i + 1}] Question: {q}")
    print(f"    Answer:   {ans}")
    print("-" * 60)

Reconfigured retriever to similarity_score_threshold (0.15) for optimal accuracy and refusals.

PART 2: BOT TRANSCRIPT

[1] Question: What is the primary defense against XSS?
    Answer:   context-aware output encoding
------------------------------------------------------------
[2] Question: How are script execution sources restricted to trusted domains?
    Answer:   Content Security Policy (CSP) headers
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='9189ccc1-0619-494e-a6fc-0edd125da0c6', metadata={'source': 'sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.54630244)), (Document(id='18d20d4c-fbfb-43d0-a3ab-4fa6f787e6bd', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast cryp

[3] Question: What is the mandatory remediation for SQL Injection?
    Answer:   parameterized queries using prepared statements
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='18d20d4c-fbfb-43d0-a3ab-4fa6f787e6bd', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast cryptographic hash functions like MD5 or SHA-256 for password storage.\nMulti-Factor Authentication (MFA) requires two or more distinct validation factors.\nSession tokens should always be stored in HttpOnly and Secure cookies to mitigate token theft via scripts.\nRate limiting must be placed on login endpoints to stop credential stuffing attacks.'), np.float32(0.16433418)), (Document(id='c51884c4-69fb-4d7d-9463-cd3203d3ec3c', metadata={'source': 'sample_data/my_kb/csrf_defenses.txt'}, page_content='Cross-Site Request Forgery (CSRF) tricks an authenticated browser into submitting unauthorized requests.\nAnti-CSRF sync

[4] Question: Which password hashing algorithms are recommended?
    Answer:   Argon2id or bcrypt
------------------------------------------------------------
[5] Question: Where should session tokens be stored to mitigate script theft?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='c51884c4-69fb-4d7d-9463-cd3203d3ec3c', metadata={'source': 'sample_data/my_kb/csrf_defenses.txt'}, page_content='Cross-Site Request Forgery (CSRF) tricks an authenticated browser into submitting unauthorized requests.\nAnti-CSRF synchronizer tokens validate that state-changing requests originate from the authentic site session.\nSetting the SameSite cookie attribute to Strict or Lax blocks third-party cookie transmission on state-changing requests.\nRe-authentication should be enforced for high-risk actions such as password changes or funds transfers.'), np.float32(0.2774102)), (Document(id='0c4f6203-49ee-4a6d-8b91-06deb44ce16d', metadata={'source': 'sample_data/my_kb/xss_prevention.txt'}, page_content='Content Security Policy (CSP) headers restrict script execution sources to trusted domains.'), np.float32(-0.0029309988))]
  self.v

[6] Question: What cookie attribute blocks third-party cookie transmission?
    Answer:   Strict or Lax
------------------------------------------------------------
[7] Question: How long do you bake sourdough bread at 450 degrees?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------
[8] Question: How does an attacker exploit an integer buffer overflow in C++?
    Answer:   I cannot answer this question based on the provided documents.
------------------------------------------------------------


In [58]:
import pandas as pd
import torch

experiments = [
    {"label": "Baseline (Like-for-like)", "k": 2, "score_threshold": 0.30},
    {"label": "Stricter Threshold",      "k": 2, "score_threshold": 0.50},
    {"label": "Permissive Threshold",   "k": 2, "score_threshold": 0.15},
    {"label": "Higher K (k=4)",         "k": 4, "score_threshold": 0.30},
    {"label": "Single Chunk (k=1)",     "k": 1, "score_threshold": 0.30},
]

refusal_phrase = "cannot answer this question"
results = []

for exp in experiments:
    temp_retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": exp["k"], "score_threshold": exp["score_threshold"]}
    )

    ans_total = 0
    ans_found = 0
    unans_total = 0
    unans_refused = 0

    for item in goldset:
        question = item["question"]
        is_unanswerable = (
            item.get("should_refuse", False)
            or item.get("unanswerable", False)
            or ("refuse" in str(item.get("ground_truth", "")).lower())
        )

        docs = temp_retriever.invoke(question)

        if not docs:
            did_refuse = True
        else:
            # Query the model exactly as we do in the main rag_chain pipeline
            context = "\n\n".join(d.page_content for d in docs)
            prompt_text = (
                f"Read the following context and answer the question.\n"
                f"If the answer is not contained in the context, reply exactly: \"I cannot answer this question based on the provided documents.\"\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer:"
            )
            input_ids = tokenizer(prompt_text, return_tensors="pt", truncation=True)
            with torch.no_grad():
                out = model.generate(**input_ids, max_new_tokens=64, do_sample=False)
            res = tokenizer.decode(out[0], skip_special_tokens=True).strip()

            did_refuse = (not res) or (refusal_phrase in res.lower())

        if is_unanswerable:
            unans_total += 1
            if did_refuse:
                unans_refused += 1
        else:
            ans_total += 1
            if not did_refuse:
                ans_found += 1

    results.append({
        "Configuration": exp["label"],
        "k": exp["k"],
        "score_threshold": exp["score_threshold"],
        "Answerable_Found": f"{ans_found}/{ans_total}",
        "Unanswerable_Refused": f"{unans_refused}/{unans_total}",
        "Answerable_Recall": round(ans_found / max(1, ans_total), 2),
        "Refusal_Accuracy": round(unans_refused / max(1, unans_total), 2)
    })

df_results = pd.DataFrame(results)
print("\n" + df_results.to_markdown(index=False))

df_results.to_csv("results_langchain.csv", index=False)
print("\nSaved evaluation results to results_langchain.csv")

/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='9189ccc1-0619-494e-a6fc-0edd125da0c6', metadata={'source': 'sample_data/my_kb/sql_injection.txt'}, page_content='SQL Injection (SQLi) allows attackers to tamper with backend database queries.\nClassic SQLi involves injecting payload strings via unsanitized user input fields.\nBlind SQLi occurs when the database shows no error messages, forcing reliance on boolean checks or time delays.\nThe mandatory remediation for SQLi is parameterized queries using prepared statements.\nDatabase user accounts must enforce the principle of least privilege to minimize damage if breached.'), np.float32(0.54630244)), (Document(id='18d20d4c-fbfb-43d0-a3ab-4fa6f787e6bd', metadata={'source': 'sample_data/my_kb/auth_standards.txt'}, page_content='Modern web authentication requires strong hashing algorithms such as Argon2id or bcrypt.\nNever use fast cryp


| Configuration            |   k |   score_threshold | Answerable_Found   | Unanswerable_Refused   |   Answerable_Recall |   Refusal_Accuracy |
|:-------------------------|----:|------------------:|:-------------------|:-----------------------|--------------------:|-------------------:|
| Baseline (Like-for-like) |   2 |              0.3  | 3/6                | 2/2                    |                0.5  |                  1 |
| Stricter Threshold       |   2 |              0.5  | 2/6                | 2/2                    |                0.33 |                  1 |
| Permissive Threshold     |   2 |              0.15 | 5/6                | 2/2                    |                0.83 |                  1 |
| Higher K (k=4)           |   4 |              0.3  | 3/6                | 2/2                    |                0.5  |                  1 |
| Single Chunk (k=1)       |   1 |              0.3  | 3/6                | 2/2                    |                0.5  |             